# Exerciții aplicate în R — date AirBox — varianta profesor

Acest notebook conține exact pregătirea datelor și exercițiile din varianta
elevului. După fiecare enunț apare o soluție posibilă și o explicație scurtă a
funcțiilor și argumentelor.

Soluțiile presupun că toate celulele anterioare au fost rulate în ordine.

## 0. Pregătirea datelor

În această secțiune:

1. instalăm și încărcăm pachetele necesare;
2. descărcăm fișierul original;
3. transformăm `time` într-o variabilă de timp;
4. introducem intenționat cinci valori `NA`;
5. transformăm intenționat `co2` din numeric în string/`character`.

Aceleași rânduri și coloane sunt modificate la fiecare rulare, astfel încât
toată clasa lucrează cu aceleași date.

In [ ]:
pachete <- c("readr", "dplyr", "ggplot2")
pachete_lipsa <- pachete[
  !sapply(pachete, requireNamespace, quietly = TRUE)
]

if (length(pachete_lipsa) > 0) {
  install.packages(pachete_lipsa)
}

library(readr)
library(dplyr)
library(ggplot2)

url_date <- paste0(
  "https://raw.githubusercontent.com/",
  "dragos-geica/cursstat/main/airbox.csv"
)

download.file(url_date, "airbox.csv", mode = "wb")

airbox_original <- read_csv(
  "airbox.csv",
  col_types = cols(time = col_character())
)

airbox <- airbox_original

# Convertim timpul într-un format pe care R îl recunoaște ca dată și oră.
airbox$time <- as.POSIXct(
  airbox$time,
  format = "%Y-%m-%d %H:%M:%S",
  tz = "Europe/Bucharest"
)

# Introducem intenționat 5 valori lipsă.
airbox$humidity[c(4, 11)] <- NA_real_
airbox$pm10[c(6, 15)] <- NA_real_
airbox$temperature[20] <- NA_real_

# Transformăm intenționat CO2 din numeric în character/string.
airbox$co2 <- paste0(airbox$co2, " ppm")

cat("Datele pentru exerciții sunt pregătite.\n")

In [ ]:
# Această celulă doar confirmă dimensiunea datelor.
dim(airbox)

## Exerciții

Rezolvă exercițiile în ordinea indicată; unele folosesc obiecte create anterior.

## 1. Inspectarea datelor

Folosește trei funcții diferite pentru:

1. a afișa primele 6 rânduri;
2. a vedea structura și tipul fiecărei coloane;
3. a obține un rezumat al valorilor.

Notează o observație despre `co2` și una despre valorile lipsă.

### Soluție posibilă

In [ ]:
head(airbox, n = 6)
str(airbox)
summary(airbox)

**Cum citim codul**

- `head(airbox, n = 6)`: primul argument este obiectul analizat; `n = 6`,
  scris în interiorul parantezelor, stabilește câte rânduri sunt afișate.
- `str(airbox)`: argumentul dintre paranteze este data frame-ul a cărui
  structură o inspectăm.
- `summary(airbox)`: argumentul este data frame-ul pentru care vrem rezumatul.

Ar trebui observat că `co2` este `character`, iar `humidity`, `temperature` și
`pm10` au valori lipsă.

## 2. Verificarea tipurilor variabilelor

Verifică separat tipul variabilelor `time`, `co2`, `temperature` și `sun`.
Apoi clasifică-le conceptual ca variabile numerice, categorice sau de timp.

Nu repara încă variabilele.

### Soluție posibilă

In [ ]:
class(airbox$time)
class(airbox$co2)
class(airbox$temperature)
class(airbox$sun)

`class(x)` primește un singur argument, obiectul `x` al cărui tip vrem să-l
verificăm. Notația `airbox$co2` înseamnă coloana `co2` din data frame-ul
`airbox`.

Conceptual, `time` este timp, `temperature` este numerică continuă, `co2` ar
trebui să fie numerică dar este stocată greșit ca text, iar `sun` este o
variabilă binară care poate fi tratată drept categorială.

## 3. Detectarea valorilor lipsă

Numără valorile `NA` din fiecare coloană. Identifică:

- coloanele care conțin valori lipsă;
- numărul de valori lipsă din fiecare;
- numărul total de valori lipsă din setul de date.

### Soluție posibilă

In [ ]:
na_pe_coloana <- colSums(is.na(airbox))
na_pe_coloana

na_total <- sum(is.na(airbox))
na_total

- `is.na(airbox)` primește data frame-ul ca argument și produce `TRUE` acolo
  unde găsește `NA`.
- `colSums(...)` primește rezultatul logic în interiorul parantezelor și adună
  valorile `TRUE` separat pentru fiecare coloană.
- `sum(...)` adună toate valorile `TRUE`, indiferent de coloană.

Rezultatul corect este: 2 `NA` în `humidity`, 1 în `temperature`, 2 în `pm10`,
în total 5.

## 4. Repararea tipurilor

Repară cele două probleme de tip:

1. transformă `co2` din texte precum `"465 ppm"` în valori numerice;
2. transformă `sun` într-un factor.

Verifică tipurile după conversie și afișează nivelurile variabilei `sun`.

### Soluție posibilă

In [ ]:
airbox$co2 <- parse_number(airbox$co2)
airbox$sun <- as.factor(airbox$sun)

class(airbox$co2)
class(airbox$sun)
levels(airbox$sun)

- `parse_number(x)` primește textul `x`, extrage partea numerică și ignoră
  unitatea `ppm`. Rezultatul este atribuit înapoi aceleiași coloane cu `<-`.
- `as.factor(x)` primește valorile ce trebuie transformate în categorii.
- `class(x)` verifică tipul rezultat, iar `levels(x)` afișează categoriile unui
  factor.

După reparare, `co2` trebuie să fie numeric, iar `sun` trebuie să fie factor cu
nivelurile `0` și `1`.

## 5. Efectul valorilor `NA` asupra mediei

Calculează media umidității:

1. fără să specifici cum sunt tratate valorile lipsă;
2. eliminând valorile lipsă din calcul.

Compară rezultatele și explică de ce diferă.

### Soluție posibilă

In [ ]:
mean(airbox$humidity)
mean(airbox$humidity, na.rm = TRUE)

`mean(x, na.rm = TRUE)` are argumentul principal `x`, vectorul numeric, urmat
de argumentul numit `na.rm`. Acesta se scrie în interiorul parantezelor după
virgulă. `TRUE` înseamnă că valorile lipsă sunt eliminate numai din calcul.

Prima comandă întoarce `NA`; a doua folosește cele 22 de valori disponibile și
produce o medie de aproximativ **43,03%**.

## 6. Filtrarea și selectarea datelor

Creează obiectul `co2_peste_460` care să conțină numai măsurătorile cu
`co2 > 460`. Păstrează doar coloanele `time`, `co2` și `temperature`, apoi
afișează rezultatul.

Câte măsurători respectă această condiție?

### Soluție posibilă

In [ ]:
co2_peste_460 <- airbox |>
  filter(co2 > 460) |>
  select(time, co2, temperature)

co2_peste_460
nrow(co2_peste_460)

- `filter(co2 > 460)` primește condiția în interiorul parantezelor și păstrează
  numai rândurile pentru care condiția este `TRUE`.
- `select(time, co2, temperature)` primește numele coloanelor ce trebuie
  păstrate, în ordinea dorită.
- `|>` trimite rezultatul din stânga către funcția din dreapta.
- `nrow(x)` primește tabelul `x` și numără rândurile sale.

Condiția este îndeplinită de **8 măsurători**.

## 7. Crearea unei variabile categoriale

Creează în `airbox` o variabilă numită `co2_nivel`:

- `"ridicat"` dacă `co2 > 460`;
- `"obisnuit"` în rest.

Transform-o în factor și construiește un tabel de frecvențe.

### Soluție posibilă

In [ ]:
airbox <- airbox |>
  mutate(
    co2_nivel = ifelse(co2 > 460, "ridicat", "obisnuit"),
    co2_nivel = as.factor(co2_nivel)
  )

table(airbox$co2_nivel)

- `mutate(...)` primește în interiorul parantezelor una sau mai multe definiții
  de coloane noi sau modificate.
- `ifelse(test, yes, no)` are trei argumente, în această ordine: condiția,
  valoarea folosită când condiția este adevărată și valoarea folosită când este
  falsă.
- `as.factor(x)` transformă rezultatul textual în categorii.
- `table(x)` numără aparițiile fiecărei categorii.

Rezultatul este **8** valori `ridicat` și **16** valori `obisnuit`.

## 8. Media, mediana și modul

Calculează:

1. media și mediana valorilor `co2`;
2. modul statistic al variabilei `sun`.

Nu folosi `mode()` pentru modul statistic. Compară media și mediana pentru
`co2`.

### Soluție posibilă

In [ ]:
media_co2 <- mean(airbox$co2, na.rm = TRUE)
mediana_co2 <- median(airbox$co2, na.rm = TRUE)

mod_statistic <- function(x) {
  frecvente <- table(x)
  names(frecvente)[which.max(frecvente)]
}

mod_sun <- mod_statistic(airbox$sun)

media_co2
mediana_co2
mod_sun

- `mean(x, na.rm = TRUE)` și `median(x, na.rm = TRUE)` primesc vectorul în
  primul argument și opțiunea pentru `NA` după virgulă.
- `function(x)` definește o funcție cu argumentul `x`.
- `table(x)` calculează frecvențele, `which.max(frecvente)` găsește poziția
  frecvenței maxime, iar `names(...)` returnează valoarea corespunzătoare.

Media CO₂ este aproximativ **452,46 ppm**, mediana este **455,5 ppm**, iar
modul lui `sun` este **1**. `mode()` din R descrie tipul intern al obiectului,
nu valoarea cea mai frecventă.

## 9. Varianța și deviația standard

Calculează varianța și deviația standard pentru `pm10`, ignorând valorile
lipsă. Verifică apoi dacă deviația standard este rădăcina pătrată a varianței.

### Soluție posibilă

In [ ]:
varianta_pm10 <- var(airbox$pm10, na.rm = TRUE)
deviatia_pm10 <- sd(airbox$pm10, na.rm = TRUE)

varianta_pm10
deviatia_pm10
sqrt(varianta_pm10)

- `var(x, na.rm = TRUE)` și `sd(x, na.rm = TRUE)` primesc vectorul numeric
  `x`; argumentul `na.rm` se pune după virgulă pentru a exclude valorile lipsă.
- `sqrt(x)` primește numărul căruia îi calculăm rădăcina pătrată.

Varianța este aproximativ **0,195 (µg/m³)²**, iar deviația standard este
aproximativ **0,441 µg/m³**.

## 10. Histogramă și boxplot

Construiește:

1. o histogramă pentru `co2`, cu 8 intervale;
2. un boxplot pentru `pm10`.

Adaugă titlu și etichete pentru axe. Alege culori diferite pentru cele două
grafice.

### Soluție posibilă

In [ ]:
ggplot(airbox, aes(x = co2)) +
  geom_histogram(
    bins = 8,
    fill = "#2C7FB8",
    color = "white"
  ) +
  labs(
    title = "Distribuția valorilor CO₂",
    x = "CO₂ (ppm)",
    y = "Frecvență"
  ) +
  theme_minimal()

ggplot(airbox, aes(y = pm10)) +
  geom_boxplot(
    fill = "#7FCDBB",
    color = "#225EA8",
    na.rm = TRUE
  ) +
  labs(
    title = "Distribuția PM10",
    x = NULL,
    y = "PM10 (µg/m³)"
  ) +
  theme_minimal()

- `ggplot(data, aes(...))`: primul argument este data frame-ul; `aes()` conține
  mapările coloanelor pe axe.
- `geom_histogram(bins = 8, fill = ..., color = ...)`: `bins` stabilește
  numărul intervalelor, `fill` culoarea interiorului, iar `color` conturul.
- `geom_boxplot(..., na.rm = TRUE)`: `na.rm` elimină valorile lipsă numai din
  grafic.
- `labs(title = ..., x = ..., y = ...)` primește etichetele ca argumente
  numite; `theme_minimal()` schimbă tema graficului.

## 11. Evoluția CO₂ în timp

Construiește un line graph pentru evoluția `co2` în funcție de `time`.
Suprapune punctele măsurate și adaugă titlu și unități pe axe.

### Soluție posibilă

In [ ]:
ggplot(airbox, aes(x = time, y = co2)) +
  geom_line(color = "#225EA8", linewidth = 0.9) +
  geom_point(color = "#D95F0E", size = 2) +
  labs(
    title = "Evoluția CO₂ în timp",
    x = "Momentul măsurării",
    y = "CO₂ (ppm)"
  ) +
  theme_minimal()

În `aes(x = time, y = co2)`, argumentele numite stabilesc variabila orizontală
și variabila verticală. `geom_line(color = ..., linewidth = ...)` desenează
linia și îi controlează culoarea și grosimea. `geom_point(color = ..., size =
...)` adaugă observațiile și stabilește culoarea și mărimea punctelor.

## 12. Bar chart pentru o variabilă categorială

Construiește un bar chart care compară numărul măsurătorilor din categoriile
`co2_nivel`. Adaugă un titlu și etichete clare.

### Soluție posibilă

In [ ]:
ggplot(airbox, aes(x = co2_nivel, fill = co2_nivel)) +
  geom_bar(show.legend = FALSE) +
  scale_fill_manual(
    values = c("obisnuit" = "#7FCDBB", "ridicat" = "#D95F0E")
  ) +
  labs(
    title = "Numărul măsurătorilor după nivelul CO₂",
    x = "Nivel CO₂",
    y = "Număr de măsurători"
  ) +
  theme_minimal()

- În `aes(x = co2_nivel, fill = co2_nivel)`, aceeași categorie stabilește atât
  poziția barelor, cât și culoarea lor.
- `geom_bar()` numără automat observațiile; `show.legend = FALSE` ascunde
  legenda redundantă.
- `scale_fill_manual(values = c(...))` primește un vector care asociază fiecărui
  nivel o culoare.

## 13. Scatterplot și corelație

Analizează relația dintre `temperature` și `humidity`:

1. construiește un scatterplot;
2. adaugă o linie de tendință liniară;
3. calculează corelația Pearson folosind numai observațiile complete;
4. interpretează direcția și intensitatea, fără să afirmi cauzalitate.

### Soluție posibilă

In [ ]:
ggplot(airbox, aes(x = temperature, y = humidity)) +
  geom_point(color = "#2C7FB8", size = 3, na.rm = TRUE) +
  geom_smooth(
    method = "lm",
    se = FALSE,
    color = "#D95F0E",
    na.rm = TRUE
  ) +
  labs(
    title = "Temperatura și umiditatea",
    x = "Temperatură (°C)",
    y = "Umiditate (%)"
  ) +
  theme_minimal()

r_temp_umiditate <- cor(
  airbox$temperature,
  airbox$humidity,
  use = "complete.obs"
)

r_temp_umiditate

- `geom_smooth(method = "lm", se = FALSE)` folosește un model liniar (`lm`) și
  nu afișează banda de incertitudine (`se`).
- `cor(x, y, use = "complete.obs")` primește cele două variabile în primele
  două poziții; argumentul `use` se pune după ele și păstrează doar perechile
  complete.

Rezultatul este aproximativ **r = −0,99**, o asociere liniară negativă foarte
puternică în acest eșantion. Nu putem concluziona că temperatura cauzează
scăderea umidității.

## 14. Probabilitate folosind distribuția normală

Folosește media și deviația standard observate pentru `temperature` ca
parametri ai unui model normal. Calculează:

1. probabilitatea modelată ca temperatura să fie cel mult 30°C;
2. probabilitatea modelată ca temperatura să fie între 20°C și 30°C.

Precizează de ce acestea sunt probabilități ale modelului, nu frecvențe
observate direct în cele 24 de rânduri.

### Soluție posibilă

In [ ]:
mu_temp <- mean(airbox$temperature, na.rm = TRUE)
sigma_temp <- sd(airbox$temperature, na.rm = TRUE)

pana_la_30 <- pnorm(
  q = 30,
  mean = mu_temp,
  sd = sigma_temp
)

intre_20_si_30 <- pnorm(
  q = 30,
  mean = mu_temp,
  sd = sigma_temp
) - pnorm(
  q = 20,
  mean = mu_temp,
  sd = sigma_temp
)

pana_la_30
intre_20_si_30

`pnorm(q, mean, sd)` calculează probabilitatea acumulată până la pragul `q`
într-o distribuție normală cu media și deviația standard specificate.
Argumentele sunt scrise în interiorul parantezelor; aici sunt numite explicit,
deci ordinea lor nu mai este esențială.

Rezultatele sunt aproximativ **0,676** și **0,515**. Ele provin din curba
normală definită de parametri; nu reprezintă simpla proporție a rândurilor care
respectă condiția și depind de presupunerea de normalitate.

## 15. Mini-analiză finală și export

Realizează o mini-analiză care să conțină:

1. media și deviația standard pentru `co2`;
2. corelația dintre `temperature` și `humidity`;
3. o concluzie de 2–3 propoziții care menționează perioada analizată,
   valorile obținute și limita privind cauzalitatea;
4. exportarea datelor reparate ca `airbox_curatat.csv`.

### Soluție posibilă

In [ ]:
media_co2 <- mean(airbox$co2, na.rm = TRUE)
deviatia_co2 <- sd(airbox$co2, na.rm = TRUE)

r_temp_umiditate <- cor(
  airbox$temperature,
  airbox$humidity,
  use = "complete.obs"
)

media_co2
deviatia_co2
r_temp_umiditate

write_csv(airbox, "airbox_curatat.csv", na = "")
list.files()

**Exemplu de concluzie:** În măsurătorile realizate pe 29 iulie 2026, nivelul
mediu al CO₂ a fost de aproximativ **452,46 ppm**, cu o deviație standard de
**13,61 ppm**. Temperatura și umiditatea au avut o corelație Pearson de
aproximativ **−0,99**, ceea ce indică o asociere liniară negativă foarte
puternică în acest eșantion, dar nu demonstrează că una dintre variabile o
cauzează pe cealaltă.

`write_csv(x, file, na = "")` primește data frame-ul `x` în prima poziție,
numele fișierului în a doua, iar `na` stabilește cum sunt scrise valorile lipsă.
`list.files()` nu are nevoie de argument aici și afișează fișierele din folderul
curent al sesiunii Colab.

## Verificare finală

Ai rulat notebook-ul de sus în jos și ai păstrat concluziile proporționale cu dovezile disponibile?